# Teste de R² por linha e modelo

Diagnóstico separado, solicitado em 06/10/2026. Usa as previsões temporais já salvas do benchmark de 2023–2025, sem novo treinamento. As coortes são as mesmas da Tabela 3: 400 datas na linha-base e 415 na expandida. Inclui os modelos individuais, comitês e regras de referência disponíveis em cada linha.

**R² = 1 − Σ(y − previsão)² / Σ(y − média de y)².** O valor 1 corresponde a previsões perfeitas; zero corresponde ao erro quadrático de uma previsão constante igual à média do consumo do conjunto avaliado; valores negativos representam erro maior que essa referência matemática. Essa média usa o conjunto de avaliação e não é uma regra operacional disponível antecipadamente. R² não é percentual de acerto nem medida de desperdício.

O R² global é calculado diretamente sobre todas as previsões da coorte, não pela média dos R² de cada semana. Os resultados por ano também são apresentados. Diferenças entre linhas não isolam a contribuição das fontes, pois as datas diferem.


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'resultados/linhas').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Execute a partir do workspace F:/PRODUTO_MESTRADO.')
OUT = ROOT / 'resultados/teste_r2_linhas_modelos_20261006'
LINES = {
    'Base': ('consumo_cardapio_calendario', 400),
    'Expandida': ('reservas_consumo_cardapio_calendario_clima', 415),
}
def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
protected = list((ROOT/'notebook').rglob('*.ipynb'))
protected += list((ROOT/'dissertacao').glob('*.docx'))
protected += list((ROOT/'resultados').rglob('*congelad*.csv'))
hashes_before = {str(p.relative_to(ROOT)): sha(p) for p in protected}
inputs = {}
overall_rows, annual_rows, cohorts, coverage_rows = [], [], [], []
print('Linha-base: consumo histórico, cardápio e calendário.')
print('Linha expandida: base acrescida de reservas e clima.')


Linha-base: consumo histórico, cardápio e calendário.
Linha expandida: base acrescida de reservas e clima.


In [2]:
def score(group):
    y = group.consumo_real.to_numpy(float)
    prediction = group.previsao.to_numpy(float)
    assert np.isfinite(y).all() and np.isfinite(prediction).all()
    error = prediction - y
    sse = float(np.square(error).sum())
    sst = float(np.square(y - y.mean()).sum())
    return {
        'n_datas': len(y), 'R2': 1 - sse/sst if len(y) >= 2 and sst > 0 else np.nan,
        'MAE': float(np.abs(error).mean()), 'RMSE': float(np.sqrt(sse/len(y))),
        'WAPE_pct': float(100*np.abs(error).sum()/np.abs(y).sum()) if np.abs(y).sum() else np.nan,
        'SSE': sse, 'SST': sst, 'media_consumo': float(y.mean()),
    }

for line, (line_id, expected_n) in LINES.items():
    folder = ROOT / 'resultados/linhas' / line_id / '04_competicao_modelos_ensembles'
    prediction_path = folder / 'previsoes_modelos_ensembles_baselines.csv'
    metric_path = folder / 'metricas_pareadas_gerais.csv'
    inputs.update({str(p.relative_to(ROOT)): sha(p) for p in [prediction_path, metric_path]})
    predictions = pd.read_csv(prediction_path, parse_dates=['data', 'origem'])
    original = pd.read_csv(metric_path)
    assert not predictions.duplicated(['modelo', 'data']).any()
    assert predictions.data.gt(predictions.origem).all()
    assert set(predictions.modelo) == set(original.modelo)
    sets = [set(g.data) for _, g in predictions.groupby('modelo')]
    dates = set.intersection(*sets)
    assert len(dates) == expected_n, (line, len(dates))
    common = predictions.loc[predictions.data.isin(dates)].copy()
    assert common.groupby('data').consumo_real.nunique().eq(1).all()
    cohorts.extend({'linha': line, 'data': d} for d in sorted(dates))
    line_rows = []
    for model, group in common.groupby('modelo'):
        assert len(group) == expected_n
        row = dict(linha=line, modelo=model, familia=group.familia.iloc[0], **score(group))
        line_rows.append(row)
        overall_rows.append(row)
        coverage_rows.append(dict(linha=line, modelo=model,
                                  n_disponiveis=int(predictions.loc[predictions.modelo.eq(model), 'data'].nunique()),
                                  n_pareadas=len(group)))
        for year, year_group in group.groupby(group.data.dt.year):
            annual_rows.append(dict(linha=line, ano=int(year), modelo=model,
                                    familia=group.familia.iloc[0], **score(year_group)))
    computed = pd.DataFrame(line_rows).merge(original, on='modelo', suffixes=('', '_original'), validate='one_to_one')
    for column in ['n_datas', 'MAE', 'RMSE', 'WAPE_pct']:
        assert np.allclose(computed[column], computed[column+'_original'], rtol=1e-9, atol=1e-9)
    print(f'{line}: {len(dates)} datas, {len(line_rows)} métodos; MAE, RMSE e WAPE conferidos.')

overall = pd.DataFrame(overall_rows).sort_values(['linha', 'MAE']).reset_index(drop=True)
annual = pd.DataFrame(annual_rows).sort_values(['linha', 'ano', 'MAE']).reset_index(drop=True)
display(overall[['linha', 'modelo', 'familia', 'n_datas', 'R2', 'MAE', 'RMSE', 'WAPE_pct']].round(4))


Base: 400 datas, 33 métodos; MAE, RMSE e WAPE conferidos.
Expandida: 415 datas, 33 métodos; MAE, RMSE e WAPE conferidos.


,linha,modelo,familia,n_datas,R2,MAE,RMSE,WAPE_pct
0,Base,CatBoost_native,boosting,400,0.3328,23.8935,33.5147,17.2314
1,Base,ExtraTrees,tree,400,0.3056,24.0113,34.1903,17.3163
2,Base,Ensemble_top3_validacao_media,ensemble,400,0.2992,24.0976,34.3482,17.3786
3,Base,Ensemble_top5_validacao_mediana,ensemble,400,0.3020,24.1787,34.2787,17.4371
4,Base,Ensemble_top5_validacao_media,ensemble,400,0.3036,24.2009,34.2385,17.4531
5,Base,Ensemble_top5_ponderado_validacao,ensemble,400,0.3035,24.2022,34.2418,17.4540
6,Base,BayesianRidge,linear,400,0.3549,24.4312,32.9538,17.6192
7,Base,KNN,distance,400,0.2919,24.4615,34.5243,17.6410
8,Base,Ensemble_ponderado_validacao,ensemble,400,0.3001,24.4619,34.3255,17.6413
9,Base,GradientBoosting,boosting,400,0.2433,24.6699,35.6899,17.7913


## Modelos da Tabela 3

Abaixo estão os cinco métodos de menor MAE em cada linha. A classificação segue o MAE da dissertação; o R² é uma informação adicional. Como o R² depende dos erros ao quadrado, sua classificação acompanha o RMSE quando os modelos usam os mesmos consumos nas mesmas datas, e pode diferir da classificação por MAE.


In [3]:
top5 = overall.groupby('linha', sort=False).head(5)
display(top5[['linha', 'modelo', 'n_datas', 'MAE', 'RMSE', 'WAPE_pct', 'R2']].round(4))


,linha,modelo,n_datas,MAE,RMSE,WAPE_pct,R2
0,Base,CatBoost_native,400,23.8935,33.5147,17.2314,0.3328
1,Base,ExtraTrees,400,24.0113,34.1903,17.3163,0.3056
2,Base,Ensemble_top3_validacao_media,400,24.0976,34.3482,17.3786,0.2992
3,Base,Ensemble_top5_validacao_mediana,400,24.1787,34.2787,17.4371,0.3020
4,Base,Ensemble_top5_validacao_media,400,24.2009,34.2385,17.4531,0.3036
33,Expandida,BayesianRidge,415,21.8109,29.9294,15.7774,0.4538
34,Expandida,Ensemble_ponderado_validacao,415,22.0816,30.3505,15.9733,0.4383
35,Expandida,Ensemble_top5_ponderado_validacao,415,22.1941,31.2057,16.0547,0.4062
36,Expandida,Ensemble_top5_validacao_media,415,22.1992,31.2052,16.0583,0.4062
37,Expandida,ExtraTrees,415,22.2688,31.7516,16.1087,0.3853


## Resultados anuais

A variabilidade do consumo muda entre anos; por isso, a comparação de R² entre períodos exige cautela. O resultado global não substitui a análise anual. Estes R² foram calculados nas datas de cada ano que pertencem à coorte comum da respectiva linha.


In [4]:
display(annual[['linha', 'ano', 'modelo', 'n_datas', 'R2', 'MAE', 'RMSE']].round(4))


,linha,ano,modelo,n_datas,R2,MAE,RMSE
0,Base,2023,GradientBoosting,113,-0.0460,19.1660,31.0910
1,Base,2023,mediana_ultimos_8_mesmo_dia_semana,113,-0.0339,19.6283,30.9117
2,Base,2023,KNN,113,-0.0607,19.7527,31.3092
3,Base,2023,SARIMAX,113,0.0585,19.8178,29.4975
4,Base,2023,XGBoost,113,-0.0084,19.9361,30.5278
5,Base,2023,SVR_RBF,113,-0.0328,20.0560,30.8939
6,Base,2023,CatBoost_native,113,-0.0104,20.0784,30.5581
7,Base,2023,Ensemble_top5_validacao_mediana,113,-0.0369,20.1191,30.9554
8,Base,2023,Ensemble_top3_validacao_media,113,-0.0820,20.6068,31.6223
9,Base,2023,Ensemble_top5_validacao_media,113,-0.0537,20.6223,31.2054


In [5]:
OUT.mkdir(parents=True, exist_ok=True)
overall.to_csv(OUT/'r2_todos_modelos.csv', index=False, encoding='utf-8-sig')
annual.to_csv(OUT/'r2_por_ano.csv', index=False, encoding='utf-8-sig')
top5.to_csv(OUT/'r2_metodos_tabela3.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(cohorts).to_csv(OUT/'datas_avaliadas.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(coverage_rows).to_csv(OUT/'cobertura_por_modelo.csv', index=False, encoding='utf-8-sig')
assert all(sha(ROOT/path) == value for path, value in hashes_before.items())
assert all(sha(ROOT/path) == value for path, value in inputs.items())
protocol = {
    'data': '2026-10-06', 'formula': 'R2 = 1 - SSE/SST; SST em torno da média dos alvos da coorte',
    'coortes': {'Base': 400, 'Expandida': 415}, 'modelos_por_linha': overall.groupby('linha').size().to_dict(),
    'calculo': 'Previsões contínuas fora do treino, já produzidas na avaliação temporal de 2023–2025.',
    'novo_treino': False, 'media_de_r2_semanais': False,
    'verificacoes': ['Sem duplicatas modelo/data', 'Alvos posteriores à origem', 'Consumos iguais entre métodos na mesma data',
                     'Reprodução de n, MAE, RMSE e WAPE do benchmark', 'Fontes, notebooks anteriores, DOCX e emissões congeladas preservados'],
    'fontes_sha256': inputs,
    'limites': 'R2 de avaliação retrospectiva; não é percentual de acerto, efeito causal ou resultado operacional. Coortes distintas entre linhas.',
}
(OUT/'protocolo.json').write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding='utf-8')
report = ['# Teste de R² por linha e modelo', '', protocol['formula'], '',
          top5[['linha','modelo','n_datas','MAE','RMSE','R2']].round(4).to_string(index=False), '',
          protocol['limites'], '', 'Resultados completos: r2_todos_modelos.csv. Resultados anuais: r2_por_ano.csv.', '',
          'Verificações concluídas: métricas do benchmark reproduzidas e arquivos originais preservados.']
(OUT/'RELATORIO.md').write_text('\n'.join(report), encoding='utf-8')
print('Resultados salvos em:', OUT)
print('Verificações concluídas. Arquivos originais preservados.')


Resultados salvos em: F:\PRODUTO_MESTRADO\resultados\teste_r2_linhas_modelos_20261006
Verificações concluídas. Arquivos originais preservados.
